# 02 — Particionamento em 3 folds (treino / validação / teste)
Implementa o esquema dos slides (31–33) e do notebook de referência, com o código em `src/split_data.py`.

1. Divide-se cada classe em **3 partes estratificadas** (3 folds).
2. Rotação: fold 1 → treino=1, validação=2, teste=3; fold 2 → treino=2, validação=3, teste=1; fold 3 → treino=3, validação=1, teste=2.
3. **Oversampling** (repetição aleatória da classe minoritária) **só em treino e validação**, até 50%/50%.
4. **Teste**: linhas originais, sem repetição, na proporção real.
5. Cada conjunto é embaralhado.

Normalização, imputação e codificação **não** são feitas aqui: serão ajustadas depois **apenas no treino** de cada fold.

In [ ]:
# Localiza a raiz do repositório (funciona no VS Code e no Colab) e a coloca no sys.path
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Raiz do projeto:", ROOT)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from src import config as cfg
from src.data_loader import load_raw
from src.split_data import (build_partitions, verify_partitions, count_cross_set_duplicates,
                            save_partitions, load_partition, load_ids)

cfg.ensure_dirs()
df = load_raw()
print("SEED =", cfg.SEED, "| K =", cfg.K_FOLDS, "| rotação:", {f: cfg.roles_for_fold(f) for f in range(1, cfg.K_FOLDS + 1)})

## 1. Construir as partições

In [ ]:
parts = build_partitions(df)

## 2. Verificações (levantam erro se algo estiver errado)
Sem sobreposição entre conjuntos, cobertura total, teste sem repetição e na proporção original, treino/validação balanceados, valores originais preservados.

In [ ]:
resumo = verify_partitions(parts, df)
resumo.to_csv(cfg.REPORTS_TABLES / "particionamento_resumo.csv", index=False)
print("Todas as verificações passaram.")
resumo

In [ ]:
fig, axes = plt.subplots(1, cfg.K_FOLDS, figsize=(12, 3.5), sharey=True)
for ax, (fold, g) in zip(axes, resumo.groupby("fold")):
    g = g.set_index("conjunto").loc[list(cfg.SET_NAMES)]
    g[["churn_No", "churn_Yes"]].plot.bar(stacked=True, ax=ax, color=["tab:blue", "tab:red"], rot=0, legend=False)
    ax.set_title(f"Fold {fold} como treino"); ax.set_xlabel("")
axes[0].set_ylabel("linhas"); axes[-1].legend(["No", "Yes"], title="Churn")
fig.suptitle("Composição de cada conjunto (treino/validação balanceados; teste com proporção original)")
fig.tight_layout(); fig.savefig(cfg.REPORTS_FIGURES / "02_composicao_conjuntos.png", dpi=150); plt.show()

## 3. Linhas idênticas em conjuntos diferentes (informativo)
O dataset não tem ID de cliente e contém perfis idênticos. Isso **não é sobreposição de registros** (os `id_origem` são disjuntos, já verificado), mas é bom saber a magnitude.

In [ ]:
count_cross_set_duplicates(parts)

## 4. Salvar em `data/processed/`
Gera `X_*`/`y_*` por fold, `ids_origem.csv` (rastreia cada linha, inclusive as repetidas) e `manifesto.json` (seed, md5 do CSV, formas).

In [ ]:
save_partitions(parts)
sorted(p.name for p in cfg.DATA_PROCESSED.iterdir())

## 5. Conferir recarregando do disco

In [ ]:
X, y = load_partition(1, "treino")
print("Fold 1 treino:", X.shape, "| % churn:", round(100 * y.mean(), 2))
X, y = load_partition(1, "teste")
print("Fold 1 teste :", X.shape, "| % churn:", round(100 * y.mean(), 2))
load_ids().groupby(["fold", "conjunto"]).agg(linhas=("id_origem", "size"), unicos=("id_origem", "nunique"), repetidas=("repetido", "sum"))

**Atenção para o grupo:** a semente (`SEED` em `src/config.py`) e os arquivos de `data/processed/` são versionados no Git para que **todos usem exatamente os mesmos folds**. Não regenere com outra semente depois que os experimentos começarem.